In [ ]:
import pandas as pd
import numpy as np

INPUT_FILE = "digikala_comments_clean_ids.csv"
TEST_SIZE = 100_000

df = pd.read_csv(INPUT_FILE)

df["text"] = df["text"].fillna("").astype(str).str.strip()
df["recommendation_status_clean"] = (
    df["recommendation_status_clean"].astype("string").str.strip()
)

df = df[
    df["text"].ne("") &
    df["recommendation_status_clean"].notna() &
    df["product_id"].notna()
].copy()

df = df.drop_duplicates(
    subset=["id", "product_id", "text", "recommendation_status_clean"]
).reset_index(drop=True)

classes = sorted(df["recommendation_status_clean"].unique())

class_counts = df["recommendation_status_clean"].value_counts()
target_balanced = {
    classes[0]: TEST_SIZE // 3 + 1,
    classes[1]: TEST_SIZE // 3,
    classes[2]: TEST_SIZE // 3
}

target_natural = (
    class_counts.reindex(classes) / len(df) * TEST_SIZE
).round().astype(int).to_dict()

difference = TEST_SIZE - sum(target_natural.values())
target_natural[classes[0]] += difference

product_counts = (
    df.groupby(["product_id", "recommendation_status_clean"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=classes, fill_value=0)
)

def select_products(available_ids, targets, seed):
    rng = np.random.default_rng(seed)
    shuffled_ids = np.array(list(available_ids), dtype=object)
    rng.shuffle(shuffled_ids)

    selected_ids = []
    selected_counts = {c: 0 for c in classes}

    for product_id in shuffled_ids:
        selected_ids.append(product_id)

        for c in classes:
            selected_counts[c] += int(product_counts.loc[product_id, c])

        if all(selected_counts[c] >= targets[c] for c in classes):
            break

    if not all(selected_counts[c] >= targets[c] for c in classes):
        raise ValueError("Not enough products to reach the requested targets.")

    return set(selected_ids)

def make_dataset(selected_ids, targets, seed):
    rng = np.random.default_rng(seed)
    selected_df = df[df["product_id"].isin(selected_ids)]

    samples = []

    for c in classes:
        class_df = selected_df[
            selected_df["recommendation_status_clean"] == c
        ]

        if len(class_df) < targets[c]:
            raise ValueError(f"Not enough samples for class: {c}")

        sampled_indices = rng.choice(
            class_df.index.to_numpy(),
            size=targets[c],
            replace=False
        )

        samples.append(df.loc[sampled_indices])

    result = pd.concat(samples, ignore_index=True)
    result = result.sample(frac=1, random_state=seed).reset_index(drop=True)

    return result

all_product_ids = set(product_counts.index)

balanced_test_products = select_products(
    all_product_ids,
    target_balanced,
    seed=43
)

remaining_product_ids = all_product_ids - balanced_test_products

natural_test_products = select_products(
    remaining_product_ids,
    target_natural,
    seed=45
)

df_test_balanced = make_dataset(
    balanced_test_products,
    target_balanced,
    seed=43
)

df_test_natural = make_dataset(
    natural_test_products,
    target_natural,
    seed=45
)

assert len(df_test_balanced) == TEST_SIZE
assert len(df_test_natural) == TEST_SIZE

assert set(df_test_balanced["product_id"]).isdisjoint(
    set(df_test_natural["product_id"])
)

df_test_balanced.to_csv("test_balanced.csv", index=False)
df_test_natural.to_csv("test_natural.csv", index=False)

print("Test Balanced:", df_test_balanced.shape)
print(df_test_balanced["recommendation_status_clean"].value_counts())

print("\nTest Natural:", df_test_natural.shape)
print(df_test_natural["recommendation_status_clean"].value_counts())


========== Train Balanced ==========
Total: 100000
recommendation_status_clean
no_idea            33334
recommended        33333
not_recommended    33333
Name: count, dtype: int64

Percentage:
recommendation_status_clean
no_idea            33.334
recommended        33.333
not_recommended    33.333
Name: proportion, dtype: float64

========== Test Balanced ==========
Total: 100000
recommendation_status_clean
no_idea            33334
recommended        33333
not_recommended    33333
Name: count, dtype: int64

Percentage:
recommendation_status_clean
no_idea            33.334
recommended        33.333
not_recommended    33.333
Name: proportion, dtype: float64

========== Train Natural ==========
Total: 100000
recommendation_status_clean
recommended        76025
no_idea            12252
not_recommended    11723
Name: count, dtype: int64

Percentage:
recommendation_status_clean
recommended        76.025
no_idea            12.252
not_recommended    11.723
Name: proportion, dtype: float64

==

In [ ]:


INPUT_FILE = "digikala_comments_clean_ids.csv"

df_test_balanced = pd.read_csv("test_balanced.csv")
df_test_natural = pd.read_csv("test_natural.csv")

df_base = pd.read_csv(INPUT_FILE)

df_base["text"] = df_base["text"].fillna("").astype(str).str.strip()
df_base["recommendation_status_clean"] = (
    df_base["recommendation_status_clean"].astype("string").str.strip()
)

df_base = df_base[
    df_base["text"].ne("") &
    df_base["recommendation_status_clean"].notna() &
    df_base["product_id"].notna()
].copy()

df_base = df_base.drop_duplicates(
    subset=["id", "product_id", "text", "recommendation_status_clean"]
).reset_index(drop=True)

balanced_test_products = set(df_test_balanced["product_id"].unique())
natural_test_products = set(df_test_natural["product_id"].unique())

df_train_balanced = df_base[
    ~df_base["product_id"].isin(balanced_test_products)
].copy()

df_train_natural = df_base[
    ~df_base["product_id"].isin(natural_test_products)
].copy()

assert set(df_train_balanced["product_id"]).isdisjoint(
    balanced_test_products
)

assert set(df_train_natural["product_id"]).isdisjoint(
    natural_test_products
)

df_train_balanced.to_csv("train_balanced.csv", index=False)
df_train_natural.to_csv("train_natural.csv", index=False)

datasets = {
    "Train Balanced": df_train_balanced,
    "Test Balanced": df_test_balanced,
    "Train Natural": df_train_natural,
    "Test Natural": df_test_natural
}

for name, dataset in datasets.items():
    print(f"\n{name}")
    print("Shape:", dataset.shape)
    print("Unique products:", dataset["product_id"].nunique())
    print("Unique texts:", dataset["text"].nunique())
    print(dataset["recommendation_status_clean"].value_counts())
    print(
        dataset["recommendation_status_clean"]
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

Balanced test products: 13164
Natural test products: 7844

Balanced product overlap: 0
Natural product overlap: 0

========== Train Balanced ==========
Total records: 3639394
Unique texts: 3639394
Unique products: 271834

Class distribution:
recommendation_status_clean
recommended        2771263
no_idea             443875
not_recommended     424256
Name: count, dtype: int64

Class percentages:
recommendation_status_clean
recommended        76.15
no_idea            12.20
not_recommended    11.66
Name: proportion, dtype: float64

========== Test Balanced ==========
Total records: 100000
Unique texts: 100000
Unique products: 13164

Class distribution:
recommendation_status_clean
no_idea            33334
recommended        33333
not_recommended    33333
Name: count, dtype: int64

Class percentages:
recommendation_status_clean
no_idea            33.33
recommended        33.33
not_recommended    33.33
Name: proportion, dtype: float64

========== Train Natural ==========
Total records: 380049